# Kapitel 4: Architektur von Large Language Models

- Das folgende Kapitel wird die Architektur des kleinsten GPT-2 Models beschrieben und gezeigt (124.000.000 Parameter)

- Benötigte Pakete:

In [9]:
from importlib.metadata import version

print("matplotlib version:", version("matplotlib"))
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

matplotlib version: 3.10.8
torch version: 2.11.0
tiktoken version: 0.12.0


- In Kapitel 3 wurden der Attention mechanismus auf Grund der verbesserten Darstellung mit sehr kleinen Embedding Dimensionen implementiert
- In diesem Kapitel werden die Embedding Dimensionen und Modellgrößen einem GPT-2 Model entsprechen
- Konfigurationsdetails:
    - vocab_size definiert die Anzahl der Tokens des cl100k_base Tokenizers.
    - context_length gibt die maximale Sequenzlänge an, die das Modell 
    gleichzeitig verarbeiten kann. GPT-2 wurde mit 1024 Tokens trainiert.
    - emb_dim legt fest dass jedes Token als Vektor mit 768 Dimensionen 
    dargestellt wird, wobei jede Dimension einen semantischen Aspekt des 
    Tokens kodiert.
    - n_heads definiert die Anzahl der Attention-Heads. Jeder Head betrachtet 
    den Text aus einem anderen Blickwinkel. Mit 768 ÷ 12 = 64 Dimensionen 
    pro Head.
    - n_layers gibt die Anzahl der hintereinandergeschalteten TransformerBlöcke an.
    - drop_rate schaltet während des Trainings zufällig 10% der Verbindungen 
    aus um Overfitting zu vermeiden.
    - qkv_bias deaktiviert den zusätzlichen Bias-Parameter in der Attention, 
    was dem Standard moderner Sprachmodelle entspricht.   

In [10]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vokabular Größe
    "context_length": 1024, # Kontextlänge
    "emb_dim": 768,         # Vektor Dimension
    "n_heads": 12,          # Anzahl der Attention Heads
    "n_layers": 12,         # Anzahl der Transformer Blöcke
    "drop_rate": 0.1,       # Dropout Rate
    "qkv_bias": False       # Query, Key, Value Bias
}

## 4.1 DummyGPTModel implementieren

- Im ersten Schritt implementieren wir ein grobes Gerüst eines General Pre-Trained Transformers mit Platzhaltern um die gesamtstruktur zu verstehen

In [11]:
import torch
import torch.nn as nn


class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        # Dummy Transformer Blöcke
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]) 
        
        # Dummy LayerNorm und Output-Head
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

# Forward: Token- und Positions-Embeddings berechnen, summieren, durch Dropout schicken, durch die Dummy 
# Transformer Blöcke laufen lassen, normalisieren und schließlich die Logits für die nächsten Token vorhersagen.
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg): # Dummy Transformer Block, das nichts tut.
        super().__init__()


    def forward(self, x):
      # Dummy Forward, der die Eingabe unverändert zurückgibt.
        return x
    
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # Dummy LayerNorm, das nichts tut.

    def forward(self, x):
        # Dummy Forward, der die Eingabe unverändert zurückgibt.
        return x

#### Dummy Klasse verwenden:

In [28]:
import tiktoken # Tokenizer für die Texte verwenden, um sie in Token-IDs umzuwandeln.

tokenizer = tiktoken.get_encoding("gpt2") # GPT-2 Tokenizer verwenden, da unser Dummy-Modell auf GPT-2 Konfiguration basiert.

batch = [] # Batch von Token-IDs erstellen, indem die Texte tokenisiert und in Tensoren umgewandelt werden.

txt1 = "Ich liebe Transformer ha ha" # Zwei Beispieltexte, die tokenisiert werden sollen.
txt2 = "Ich liebe Sprachmodelle" # Zwei Beispieltexte, die tokenisiert werden sollen.

batch.append(torch.tensor(tokenizer.encode(txt1))) # Text 1 tokenisieren, in einen Tensor umwandeln und zum Batch hinzufügen.
batch.append(torch.tensor(tokenizer.encode(txt2))) # Text 2 tokenisieren, in einen Tensor umwandeln und zum Batch hinzufügen.
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[   40,   354,  6486,  1350,  3602, 16354,   387,   387],
        [   40,   354,  6486,  1350,  5522,   620,  4666, 13485]])


In [29]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M) # Dummy GPT Modell mit der definierten Konfiguration erstellen.

logits = model(batch) # Batch von Token-IDs durch das Dummy-Modell laufen lassen, um die Logits für die nächsten Token vorherzusagen.


# 1. Logits haben Shape [2, 7, 50257] — pro Token 50257 Scores für die nächsten möglichen Tokens.
# 2 --> 2 Texte im Batch, 7 --> Sequenzlänge (Anzahl der Tokens pro Text), 50257 --> Vokabulargröße (Anzahl der möglichen nächsten Tokens).
print("Output shape:", logits.shape) 
print("logits:", logits) # Logits ausgeben, die die Vorhersagen für die nächsten Token repräsentieren.

# 2. Größten Score-Index aus 50257 möglichen nächsten Tokens für jedes Token in der Sequenz auswählen, um die vorhergesagten nächsten Token-IDs zu erhalten.
# Position des größten Scores = Token-ID des vorhergesagten nächsten Tokens.
predicted_ids = torch.argmax(logits, dim=-1)
print("\nPredicted TokenIDs Satz 1:", predicted_ids[0].tolist())
print("Predicted TokenIDs Satz 2:", predicted_ids[1].tolist())

# 3. Token-IDs zurück in lesbaren Text umwandeln, um die vorhergesagten nächsten Tokens zu sehen.
predicted_tokens_1 = tokenizer.decode(predicted_ids[0].tolist())
predicted_tokens_2 = tokenizer.decode(predicted_ids[1].tolist())
print("\nPredicted Tokens Satz 1:", predicted_tokens_1)
print("Predicted Tokens Satz 2:", predicted_tokens_2)








Output shape: torch.Size([2, 8, 50257])
logits: tensor([[[ 0.7336, -1.1718,  0.0319,  ..., -0.5045,  0.3875, -0.0966],
         [-0.2137, -0.5371,  0.0305,  ...,  0.9313,  0.4393,  0.8954],
         [ 0.9299, -0.7605, -0.5757,  ...,  0.4171,  1.1669, -0.5149],
         ...,
         [-0.3961,  1.1770,  0.3591,  ...,  1.0325, -1.0188, -0.1571],
         [ 0.3375,  0.4295,  1.0041,  ...,  0.0663,  0.5144,  0.8624],
         [ 0.3905,  1.5745,  0.6406,  ..., -0.0125, -0.5562,  0.7602]],

        [[ 0.6071, -0.6882, -0.3438,  ..., -0.0882,  0.4655, -0.5635],
         [-0.3174,  0.1708,  0.3470,  ...,  1.3430,  1.0098,  0.4477],
         [ 0.9303, -0.4941, -0.8053,  ...,  0.5960,  0.3554, -0.7412],
         ...,
         [-0.0517, -0.7507,  0.4866,  ...,  0.8407, -0.1186,  0.8195],
         [ 1.0671,  0.1470, -0.2110,  ...,  1.0622, -0.4368,  0.3001],
         [ 0.5463,  0.1924, -0.7899,  ...,  0.4505,  0.3756, -0.8196]]],
       grad_fn=<UnsafeViewBackward0>)

Predicted TokenIDs Satz 1: [3

- Hier kommt jetzt nichts Sinnvolles raus weil:
- Die Embeddings sind zufällig, das Modell ist untrainiert und "weiß" also nicht was die Embeddings bedeuten, die Dummy-Blöcke verändern nichts, also sind die Logits koplett zufällig
- D.h. die vorhergesagten TokenIDs sind die, die zufällig den höchsten Score erhalten haben 


## 4.1 Layer Normalisation (LayerNorm)

- LayerNorm normalisiert die Layer Activations (=Ergebnisse die an die nächste Schicht/Layer weitergegeben werden) einer Schicht auf Mittelwert 0 und Varianz 1
- Verhindert explodierende/verschwindende Werte über viele Schichten → stabiles Training
- LayerNorm wird vor und nach der Self-Attention, innerhalb des Transformer Blocks und vor der finalen Output Schicht angewendet


#### Wie funktioniert LayerNorm:

In [ ]:
# Codebeispiel: Beispielhafte Batch durch ein einfaches Layer laufen lassen und Ergebnisse mit LayerNorm vergleichen.

torch.manual_seed(123)

# Ein Batch von 2 Beispielen mit 5 
batch_example = torch.randn(2, 5) 

# Einfaches Layer mit Linear-Transformation und ReLU-Aktivierung erstellen.
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())


out = layer(batch_example) 
print("Layer Output:", out, "\n")

# Mittelwert und Varianz berechnen.
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var, "\n\n")

# LayerNorm durführen um Mittelwert 0 und Varianz 1 für jede Zeile zu erhalten. Formel: (x - mean) / sqrt(var)
out_norm = (out - mean) / torch.sqrt(var)
print("Normalized layer outputs:\n", out_norm)

mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("Mittelwert:\n", mean)
print("Varianz:\n", var)



Layer Output: tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>) 

Mean:
 tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>) 


Normalized layer outputs:
 tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
Mean:
 tensor([[-5.9605e-08],
        [ 1.9868e-08]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


#### LayerNorm in einer Klasse implementieren:

In [41]:
# LayerNorm Klasse normalisiert die Eingabe so, dass sie pro Zeile einen Mittelwert von 0 und eine Varianz von 1 hat.
class LayerNorm(nn.Module): 
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5 # Kleine Konstante, um Division durch Null zu vermeiden.


        # lernbarer Multiplikator, startet bei 1 (--> ändert erstmal nichts)
        # Das Modell kann damit lernen einzelne Dimensionen stärker oder schwächer zu gewichten
        self.scale = nn.Parameter(torch.ones(emb_dim)) 

        # lernbarer Shift, startet bei 0 (--> verschiebt erstmal nichts)
        # Das Modell kann damit lernen den Mittelwert gezielt zu verschieben
        self.shift = nn.Parameter(torch.zeros(emb_dim))



    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True) # Mittelwert pro Zeile berechnen 
        var = x.var(dim=-1, keepdim=True, unbiased=False) # Varianz pro Zeile berechnen 
        norm_x = (x - mean) / torch.sqrt(var + self.eps) # Normalisieren: (x - mean) / sqrt(var + eps)
        return self.scale * norm_x + self.shift # Skalieren und Verschieben: scale * norm_x + shift
    
    # Anmerkung: dim=-1 bedeutet, dass die Normalisierung über die letzte 
    # Dimension (die Embedding-Dimension=Reihen) erfolgt. 

#### Scale und Shift
- scale (×1) und shift (+0) sind trainierbare Parameter — zu Beginn ohne Effekt
- Werden beim Training angepasst, falls eine andere Verteilung die Performance verbessert
- eps verhindert Division durch 0, falls Varianz = 0

#### Biased Variance
- unbiased=False → Varianz wird durch n geteilt, nicht n-1 (kein Bessel's Correction)
- Bei großem n (z.B. emb_dim=768) ist der Unterschied zu n-1 vernachlässigbar
- GPT-2 wurde mit dieser Variante trainiert → wir übernehmen es für Kompatibilität mit den Pretrained Weights in Kapitel 5


#### LayerNorm Klasse verwenden:

In [46]:
# Beispiel Batch initialisieren
torch.manual_seed(123)
batch_example = torch.randn(4, 8)
print("Original Batch:\n", batch_example, "\n")

# Ausgabe von Tensoren ohne wissenschaftliche Notation, um die Werte besser lesbar zu machen.
torch.set_printoptions(sci_mode=False)

# LayerNorm auf den Batch anwenden
ln = LayerNorm(emb_dim=8) # LayerNorm mit Embedding-Dimension 8 erstellen.
normalized_batch = ln(batch_example)


print("Normalized Batch:\n", normalized_batch, "\n")
# Mittelwert und Varianz des normalisierten Batches berechnen, um zu überprüfen, dass sie 0 bzw. 1 sind.
mean = normalized_batch.mean(dim=-1, keepdim=True)
var = normalized_batch.var(dim=-1, keepdim=True, unbiased=False)
print("Mittelwert:\n", mean)
print("Varianz:\n", var)   

Original Batch:
 tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  0.3486,  0.6603, -0.2196, -0.3792],
        [ 0.7671, -1.1925,  0.6984, -1.4097,  0.1794,  1.8951,  0.4954,  0.2692],
        [-0.0770, -1.0205, -0.1690,  0.9178,  1.5810,  1.3010,  1.2753, -0.2010],
        [ 0.4965, -1.5723,  0.9666, -1.1481, -1.1589,  0.3255, -0.6315, -2.8400]]) 

Normalized Batch:
 tensor([[ 0.9296, -0.3386, -0.6482, -1.3485,  0.9572,  1.7247, -0.4417, -0.8344],
        [ 0.5521, -1.3996,  0.4836, -1.6160, -0.0333,  1.6756,  0.2815,  0.0562],
        [-0.6022, -1.6782, -0.7070,  0.5324,  1.2888,  0.9695,  0.9402, -0.7435],
        [ 1.0155, -0.7473,  1.4161, -0.3859, -0.3950,  0.8698,  0.0544, -1.8276]],
       grad_fn=<AddBackward0>) 

Mittelwert:
 tensor([[ 0.0000],
        [ 0.0000],
        [ 0.0000],
        [-0.0000]], grad_fn=<MeanBackward1>)
Varianz:
 tensor([[0.9999],
        [1.0000],
        [1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


- Die Varianz ist aufgrund der eps Konstante nicht ganz 0 